# Notebook to apply Langextract over all documents to anonymize

## Prompt and function definitions

In [1]:
import os
from dotenv import load_dotenv
import pandas as pd
import langextract as lx
from rich.pretty import pprint
import textwrap
import timeit

import re
import pandas as pd
from more_itertools import unique_justseen
from aymurai.api.endpoints.routers.misc.document_extract import extraction
from aymurai.database.utils import text_to_uuid

In [2]:
######## CONFIGURATION

DOCS_PATH = '/Users/sofi/Desktop/collectiveAI/projects/data-genero/backend/resources/data/sample/'#'/Users/sofi/Desktop/collectiveai/projects/AymurAI/backend/resources/data/sample/'

https://github.com/google/langextract/tree/main/examples/ollama 

### New prompt - long

In [15]:
PROMPT = textwrap.dedent("""
Sos un extractor de ENTIDADES sensibles para anonimización en documentos judiciales en español. Vas a leer cada parrafo con atención y extraer todas las entidades que correspondan según
las clases definidas más abajo. 

INSTRUCCIONES ESTRICTAS:
- Las iniciales de personas deben ser tomadas como clase Persona.
- Extraé SOLO spans EXACTOS que estén en el texto (no parafrasees ni infieras).
- Si una clase NO aparece, NO devuelvas nada de esa clase.
- NO inventes códigos ni números. No completes nada por contexto.
- No superpongas entidades; una mención = una extracción.
- Tené en cuenta que, si bien las fechas sensibles al caso deben sacarse, la única fecha que debe permanecer es la fecha de resolución que suele anunciarse al comienzo como
"Buenos Aires, ... "
- Si una entidad no está clara, NO la extraigas.
- Si una entidad está sutilmente mal escrita, extráela igual.
- Si una entidad está incompleta, extráela igual.
- Si una entidad está repetida, extráelas todas.
- Si una entidad está en un formato no estándar, pero detectas que corresponde a esa entendidad, extráela igual.

POSIBLES CLASES Y DESCRIPCIONES:
- BANCO: Entidad bancaria
- CBU: número de 22 dígitos asociado a una entidad bancaria
- CORREO_ELECTRONICO: dirección de email
- CUIT_CUIL: código único de identificación tributaria o laboral en Argentina (formato ##-########-#)
- CUIJ: código único de identificación judicial (formato ##-########-#)
- DIRECCION: puede presentarse como calle y altura, intersección de calles, o número de domicilio
- DNI: documento nacional de identidad (7-8 dígitos)
- EDAD: edad de una persona, puede estar en años o meses
- ESTUDIOS: nivel educativo alcanzado (primario, secundario, terciario, universitario, posgrado, doctorado), puede estar acompañado de "incompleto", "completo", "finalizado", "en curso"
- FECHA: fechas en cualquier formato (dd/mm/aaaa, dd-mm-aaaa, dd de mes de aaaa, también puede ser dos fechas juntas como por ejemplo el 5 y 7 de mayo de 2020 y similares)
- LINK: URLs o enlaces web
- LOC: nombres de localidades, provincias, países, continentes
- MARCA_AUTOMOVIL: marcas de automóviles (Ford, Chevrolet, Toyota, Renault, Fiat, etc)
- NACIONALIDAD: nacionalidades (argentina, italiana, española, uruguaya, chilena, paraguara, etc)
- NUM_CAJA_AHORRO: número de caja de ahorro o cuenta bancaria
- NUM_EXPEDIENTE: número de expediente judicial o administrativo en formato \d+/\d{4} (por ejemplo 1234/2020)
- NUM_MATRICULA: número de matrícula profesional (médica, abogacía, etc) o académica.
- PATENTE_DOMINIO: patentes o dominio de un vehículo. En Argentina, pueden ser de formato [A-Z]{3}\d{3} o [A-Z]{2}\d{3}[A-Z]{2}
- PER: Nombre y apellido(s) de una persona física. Los nombres inicializados y los apodos también cuentan como información sensible a anonimizar.                     
- NUM_ACTUACION: Número identificatorio de una actuación administrativa o contravencional.
- TELEFONO: Número telefónico (fijo o celular).
                         
Razona detenidamente antes de elegir casa entidad y asociá dicho razonamiento a una variable justification y score de confianza de tu razonamiento, luego, para todo el parrafo crea una 
salida que sea una lista de diccionarios, uno por clase extraida, con las siguientes claves:
- text: el texto exacto de la entidad
- label: la clase de la entidad (una de las listadas más arriba)
""")


In [16]:
# -----------------
# Ejemplos balanceados (judiciales)
#   1) PER/FECHA/DIRECCION/LOC
#   2) Un único ejemplo de códigos (para enseñar formato)
#   3) Datos personales típicos de actuaciones
#   4) NEGATIVO: no hay códigos -> salida vacía
# -----------------
examples = [
    lx.data.ExampleData(
        text=textwrap.dedent("""En la Ciudad Autónoma de Buenos Aires, el día 5 de mayo de 2023, "
              "el Sr. Fiscal hace saber que Juan Pérez se domicilia en la calle "
              "Sarmiento 1234, localidad de Moreno."""),
        extractions=[
            lx.data.Extraction(extraction_class="FECHA",     extraction_text="5 de mayo de 2023"),
            lx.data.Extraction(extraction_class="PER",       extraction_text="Juan Pérez"),
            lx.data.Extraction(extraction_class="DIRECCION", extraction_text="Sarmiento 1234"),
            lx.data.Extraction(extraction_class="LOC",       extraction_text="Moreno"),
        ],
    ),

    lx.data.ExampleData(
        text = textwrap.dedent(""""3) Abstenerse de ingresar y/o concurrir a la Villa 13"""),
        extractions = [
            lx.data.Extraction(extraction_class="LOC", extraction_text = "Villa 13")
        ]
    ),
    lx.data.ExampleData(
        text=textwrap.dedent("""JUZGADO NACIONAL EN LO CRIMINAL Y CORRECCIONAL N° 10 - Secretaría N° 19. "
              "Causa N° 52345/2022. CUIJ: 12-34567890-1. Actuación N° 2022-009876."""),
        extractions=[
            lx.data.Extraction(extraction_class="NUM_EXPEDIENTE", extraction_text="52345/2022"),
            lx.data.Extraction(extraction_class="CUIJ",           extraction_text="12-34567890-1"),
            lx.data.Extraction(extraction_class="NUM_ACTUACION",  extraction_text="2022-009876"),
        ],
    ),
    lx.data.ExampleData(
        text=textwrap.dedent("""Comparece Miguel Torres, DNI 30123456, de 34 años de edad, nacionalidad paraguaya, "
              "con estudios secundarios completos, con último domicilio en Av. Corrientes 3456 de esta ciudad, "
              "junto a su cuñado Jorge Pérez."""),
        extractions=[
            lx.data.Extraction(extraction_class="PER",         extraction_text="Miguel Torres"),
            lx.data.Extraction(extraction_class="DNI",         extraction_text="30123456"),
            lx.data.Extraction(extraction_class="EDAD",        extraction_text="34"),
            lx.data.Extraction(extraction_class="NACIONALIDAD",extraction_text="paraguaya"),
            lx.data.Extraction(extraction_class="ESTUDIOS",    extraction_text="estudios secundarios completos"),
            lx.data.Extraction(extraction_class="DIRECCION",   extraction_text="Av. Corrientes 3456"),
            lx.data.Extraction(extraction_class="PER",         extraction_text="Jorge Pérez"),
        ],
    ),
    lx.data.ExampleData(
        text=textwrap.dedent("""1) transferencia defondos a cuenta de terceros por $167.000,- hacia una cuenta a nombre de la Sra. Carla Analía Gonzales, CUIL 27-25011757-0, CBU 0740399088000036512321, del Banco Santander. """),
        extractions=[
            lx.data.Extraction(extraction_class="PER",         extraction_text="Carla Analía Gonzales"),
            lx.data.Extraction(extraction_class="CUIL",       extraction_text="27-25011757-0"),
            lx.data.Extraction(extraction_class="CBU",        extraction_text="0740399088000036512321"),
            lx.data.Extraction(extraction_class="BANCO",      extraction_text="Banco Santander"),
        ],
    ),

  lx.data.ExampleData(
        text=textwrap.dedent("""teléfono celular 1141504528 y dirección de correo electrónico alejandro.perezgarcia@gmail.com."""),
        extractions=[
            lx.data.Extraction(extraction_class="TELEFONO",     extraction_text="1141504528"),
            lx.data.Extraction(extraction_class="CORREO_ELECTRONICO", extraction_text="alejandro.perezgarcia@gmail.com"),
        ],
    ),
lx.data.ExampleData(
        text=textwrap.dedent("""Por otra parte, la División Investigaciones Judiciales de la Policía Federal Argentina informó que no se dio intervención a  ninguna otra Fiscalía u otro Juzgado por la sustracción del vehículo Volkswagen Voyage, dominio KXY-876 """),
   extractions=[
            lx.data.Extraction(extraction_class="PATENTE_DOMINIO", extraction_text="KXY-876"),
            lx.data.Extraction(extraction_class="MARCA_AUTOMOVIL", extraction_text="Volkswagen Voyage"),
        ],
    ),
    lx.data.ExampleData(
        text=textwrap.dedent("""Cecilia Lopez Gracia médica del Hospital Penna, Servicio SAME, M. N. 123.558."""),
        extractions=[
            lx.data.Extraction(extraction_class="PER", extraction_text="Cecilia Lopez Gracia"),
            lx.data.Extraction(extraction_class="NUM_MATRICULA", extraction_text="M. N. 123.558"),
        ],
    ),
    lx.data.ExampleData(
        text=textwrap.dedent("""A  su  vez,  requirió  informes  al  Banco BBVA   Francés, respecto  de  las  cuentas  bancarias  de  la  denunciante,  Carla Alejandra Garcia, D.N.I.  36.998.621 identificadas  como  Caja  de  ahorro  en  pesos  argentinos  número  117-59824/6  con  CBU  0180132640000004685591 y  Caja  de  ahorro  en  dólares  número  119-619018/2 con  CBU  0170115544000062081822."""),
        extractions=[
            lx.data.Extraction(extraction_class="PER",         extraction_text="Carla Alejandra Garcia"),
            lx.data.Extraction(extraction_class="DNI",         extraction_text="36.998.621"),
            lx.data.Extraction(extraction_class="NUM_CAJA_AHORRO", extraction_text="117-59824/6"),
            lx.data.Extraction(extraction_class="CBU",        extraction_text="0180132640000004685591"),
            lx.data.Extraction(extraction_class="NUM_CAJA_AHORRO", extraction_text="119-619018/2"),
            lx.data.Extraction(extraction_class="CBU",        extraction_text="0170115544000062081822"),
        ],
    ), 

    lx.data.ExampleData(
        text=textwrap.dedent("""La grabación se encuentra disponible en el link: https://jusbairess.webex.com/jusbaire/njdndsgbnsdngsnv"""),
        extractions=[
            lx.data.Extraction(extraction_class="LINK", extraction_text="https://jusbairess.webex.com/jusbaire/njdndsgbnsdngsnv"),
        ],
    ),
    lx.data.ExampleData(
        text=textwrap.dedent("""VISTOS: Que a fin de ordenar la marcha del proceso, se fija audiencia preliminar. "
              "No se consignan números de expediente, CUIJ ni domicilios en el presente proveído."""),
        extractions=[],  # ejemplo negativo: desalienta devolver clases ausentes
    ),
]

### Function definitions

In [17]:
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

def take_start_end_paragraphs(paragraphs):
    # Create start and end character positions
    start_end_chars = []
    current_pos = 0

    for i, paragraph in enumerate(paragraphs):
        start_char = current_pos
        end_char = start_char + len(paragraph)
        start_end_chars.append(
            {
                "paragraph_position": i,
                "text": paragraph,
                "paragraph_id": str(text_to_uuid(paragraph)).replace("-", ""),
                "start_char": start_char,
                "end_char": end_char,
            }
        )
        # +1 for the newline character between paragraphs (except after the last one)
        current_pos = end_char + 1

    return start_end_chars

def constract_paragraph(document):
    paragraphs = [line.strip() for line in document.split("\n") if line.strip()]
    paragraphs = [re.sub(r"\s{2,}", " ", line) for line in paragraphs]
    paragraphs = list(unique_justseen(paragraphs))
    return paragraphs

def extract_documents(df,text_col = 'text',doc_col = 'name'):
    document_texts = {}
    for doc_name in set(df[doc_col]):
        doc_df = df[df['name']==doc_name]
        doc_text = " ".join(doc_df[text_col])
        document_texts[doc_name] = doc_text
    return document_texts

def langextract_to_dict(result):
    out = []
    for i in range(len(result.extractions)):
        #paragraph = result.text
        label = result.extractions[i].extraction_class
        text = result.extractions[i].extraction_text
        start_char = result.extractions[i].char_interval.start_pos
        end_char = result.extractions[i].char_interval.end_pos
        attrs = result.extractions[i].attributes
        alignment_status = result.extractions[i].alignment_status

        out.append({
            "label": label,
            "text": text,
            "start_char": start_char,
            "end_char": end_char,
            "attrs": attrs,
            "alignment_status": alignment_status
        })
        #print(f"{i+1}: \n CLASS: {result.extractions[i].extraction_class} \n TEXT: {result.extractions[i].extraction_text} \n from_chars: {text[result.extractions[i].char_interval.start_pos:result.extractions[i].char_interval.end_pos]}")
    return out

def langextract_prediction(text,PROMPT, examples,openai_api_key):
    result =  lx.extract(
        text_or_documents=text,
        prompt_description=PROMPT,
        examples=examples,
        language_model_type=lx.inference.OpenAILanguageModel,
        model_id="gpt-4o",
        api_key=openai_api_key,
        max_char_buffer=1000,
        extraction_passes=1,
        max_workers=6,
        fence_output=True,
        use_schema_constraints=False, # https://github.com/google/langextract
        language_model_params={
            "temperature": 0.1,
            "top_p": 0.9,
            "max_tokens": 400,
            "timeout": 600,},
            debug=False)
    return result, langextract_to_dict(result)

## Load documents

In [5]:
#documents0203entrerios-docs02-08-sin05.csv
df = pd.read_csv('documents0203entrerios-docs02-08-sin05.csv')#documents-02-08-sin05.csv')
df.head()

,text,prediction,validation,id_x,created_at_x,updated_at,id_y,document_id,paragraph_id,order,created_at_y,name
0,JUZGADO DE 1RA INSTANCIA EN LO PENAL CONTRAVEN...,[],[],c09fba61330852c4813fcbb2f3bec11e,2025-08-08 19:39:59-03:00,2025-08-08 19:40:45-03:00,57f85770879c420d8de2a0f3267b653b,518a7f34ad865b91ad95d954093f091a,c09fba61330852c4813fcbb2f3bec11e,NaN,2025-08-22 21:20:53-03:00,document-02.docx
1,JUZGADO DE 1RA INSTANCIA EN LO PENAL CONTRAVEN...,[],[],c09fba61330852c4813fcbb2f3bec11e,2025-08-08 19:39:59-03:00,2025-08-08 19:40:45-03:00,768db7897dc943489e9570a117975fe1,2536b5d2111d55c6b545e6bcbb75e002,c09fba61330852c4813fcbb2f3bec11e,NaN,2025-08-27 01:00:28-03:00,document-08.docx
2,JUZGADO DE 1RA INSTANCIA EN LO PENAL CONTRAVEN...,[],[],c09fba61330852c4813fcbb2f3bec11e,2025-08-08 19:39:59-03:00,2025-08-08 19:40:45-03:00,2c74793eac2341969aa3dcce7950f43b,6a7d2422da6a5852b68a7bee678d1aae,c09fba61330852c4813fcbb2f3bec11e,NaN,2025-08-27 02:56:14-03:00,document-04.docx
3,ANTECEDENTES,[],[],adee5f8144eb5e78abb3c56121e906cd,2025-08-08 19:40:02-03:00,2025-08-08 19:40:45-03:00,b31e11b3cff24069b7cf3faa666fbdec,0aec75365ad0511698f72bacb8b88212,adee5f8144eb5e78abb3c56121e906cd,NaN,2025-08-22 19:41:55-03:00,document-03.docx
4,ANTECEDENTES,[],[],adee5f8144eb5e78abb3c56121e906cd,2025-08-08 19:40:02-03:00,2025-08-08 19:40:45-03:00,4abeb6f8ff3340cb99cfe61b16276012,518a7f34ad865b91ad95d954093f091a,adee5f8144eb5e78abb3c56121e906cd,NaN,2025-08-22 21:20:53-03:00,document-02.docx


In [6]:
set(df.name)

{'aymurai - ejemplo 02.docx',
 'aymurai - ejemplo 03.docx',
 'document-02.docx',
 'document-03.docx',
 'document-04.docx',
 'document-06.docx',
 'document-07.docx',
 'document-08.docx',
 nan}

### Prepare data

In [7]:
docs2analize = ['2','3','4','6','7','8']
docs_file = [f for f in os.listdir(DOCS_PATH) if ('.docx' in f) and (len(set(docs2analize)&set(f))==1) ]
docs_file

['document-06.docx',
 'documento-entrerios-03.docx',
 'documento-entrerios-02.docx',
 'document-07.docx',
 'document-02.docx',
 'document-03.docx',
 'document-04.docx',
 'document-08.docx']

In [8]:
documents = {}
doc_paragraphs = {}
doc_start_end_chars = {}
joined_texts = {}
for d in docs_file:
    print(d)
    path = DOCS_PATH + d
    # Extract document
    document = extraction(path)
    # Construct paragraphs
    paragraphs = constract_paragraph(document)
    start_end_chars = take_start_end_paragraphs(paragraphs)
    documents[d] = document
    doc_paragraphs[d] = paragraphs
    joined_text = "\n".join(paragraphs)
    joined_texts[d] = joined_text
    doc_start_end_chars[d] = start_end_chars
    print(start_end_chars)
    print('\n')

document-06.docx
[{'paragraph_position': 0, 'text': 'JUZGADO DE 1RA INSTANCIA EN LO PENAL CONTRAVENCIONAL Y DE FALTAS N°10 SECRETARÍA UNICA', 'paragraph_id': '45b5531cdc7c5f7c9aaaca30e8b779a4', 'start_char': 0, 'end_char': 86}, {'paragraph_position': 1, 'text': 'GONZALEZ, MARCELO SOBRE 149 BIS - AMENAZAS', 'paragraph_id': '561f1f20e55d5ad1b8ef64340d08c7f4', 'start_char': 87, 'end_char': 129}, {'paragraph_position': 2, 'text': 'Número: IPP 31972/2018-0', 'paragraph_id': '7009436c451d52c2b5090e8e252b690e', 'start_char': 130, 'end_char': 154}, {'paragraph_position': 3, 'text': 'CUIJ: IPP J-01-00059758-5/2018-0', 'paragraph_id': '1b20a1be99cb5c4684f16cf1f70cf5b6', 'start_char': 155, 'end_char': 187}, {'paragraph_position': 4, 'text': 'Actuación Nro: 12807225/2019', 'paragraph_id': 'f44b4abccf09563b9b6e263551ba938a', 'start_char': 188, 'end_char': 216}, {'paragraph_position': 5, 'text': 'ACTA DE AUDIENCIA', 'paragraph_id': 'd140d717f5b35a4195baf3b7ee81ad8c', 'start_char': 217, 'end_char': 2

In [10]:
print(joined_texts['documento-entrerios-02.docx'])

JUZGADO DE FAMILIA N.o 1 DE LA CIUDAD DE SAN LORENZO Expediente N.o 3187/2023 Carátula: Rodríguez, Ana Carolina c/ Fernández, Diego Esteban s/ Violencia Familiar
SENTENCIA
En la ciudad de San Lorenzo, Provincia de Santa Fe, a los 22 días del mes de noviembre de 2023, siendo las 09:15 horas, la Sra. Jueza de Familia Dra. Verónica Salvatierra dicta la presente resolución en los autos caratulados "Rodríguez, Ana Carolina c/ Fernández, Diego Esteban s/ Violencia Familiar", Expte. N.o 3187/2023.
I. ANTECEDENTES
Con fecha 10 de noviembre de 2023, la Sra. Ana Carolina Rodríguez, DNI 34.112.456, con domicilio en calle Belgrano 785, Barrio Centro, San Lorenzo, denunció a su cónyuge, el Sr. Diego Esteban Fernández, DNI 31.998.210, con domicilio en calle Moreno 1520, por hechos de violencia física, verbal y patrimonial.
La denunciante manifestó que conviven desde hace doce años y tienen una hija en común, M.F.R., de 9 años. Indicó que, desde hace aproximadamente cinco años, el denunciado comenzó 

In [13]:
print(documents['documento-entrerios-02.docx'])

JUZGADO DE FAMILIA N.o 1 DE LA CIUDAD DE SAN LORENZO Expediente N.o 3187/2023 Carátula: Rodríguez, Ana Carolina c/ Fernández, Diego Esteban s/ Violencia Familiar 
SENTENCIA 
En la ciudad de San Lorenzo, Provincia de Santa Fe, a los 22 días del mes de noviembre de 2023, siendo las 09:15 horas, la Sra. Jueza de Familia Dra. Verónica Salvatierra dicta la presente resolución en los autos caratulados "Rodríguez, Ana Carolina c/ Fernández, Diego Esteban s/ Violencia Familiar", Expte. N.o 3187/2023.
I. ANTECEDENTES 
		Con fecha 10 de noviembre de 2023, la Sra. Ana Carolina Rodríguez, DNI 34.112.456, con domicilio en calle Belgrano 785, Barrio Centro, San Lorenzo, denunció a su cónyuge, el Sr. Diego Esteban Fernández, DNI 31.998.210, con domicilio en calle Moreno 1520, por hechos de violencia física, verbal y patrimonial.
		La denunciante manifestó que conviven desde hace doce años y tienen una hija en común, M.F.R., de 9 años. Indicó que, desde hace aproximadamente cinco años, el denunciado c

## OpenAI

### Apply openAI + langextract

#### One doc at the time


In [ ]:
import os
from dotenv import load_dotenv

openai_api_key = os.getenv("OPENAI_API_KEY")
load_dotenv()

file = 'documento-entrerios-02.docx'
text = joined_texts[file]
predictions = {}
results = {}


results[file], predictions[file] = langextract_prediction(text, PROMPT, examples, openai_api_key)

# result = lx.extract(
#     text_or_documents=text,
#     prompt_description=PROMPT,
#     examples=examples,
#     language_model_type=lx.inference.OpenAILanguageModel,
#     model_id="gpt-4o",
#     api_key=openai_api_key,
#     max_char_buffer=1000,
#     extraction_passes=1,
#     max_workers=6,
#     fence_output=True,
#     use_schema_constraints=False, # https://github.com/google/langextract
#     language_model_params={
#         "temperature": 0.1,
#         "top_p": 0.9,
#         "max_tokens": 400,
#         "timeout": 600,
#     },
#     debug = False
# )

In [ ]:
file = 'documento-entrerios-03.docx'
text = joined_texts[file]

results[file], predictions[file] = langextract_prediction(text, PROMPT, examples, openai_api_key)


In [36]:
file = 'document-02.docx'
text = joined_texts[file]

results[file], predictions[file] = langextract_prediction(text, PROMPT, examples, openai_api_key)


/var/folders/yf/2cttyjmx5j35yhq4st33mdmm0000gn/T/ipykernel_45695/3237458610.py:63: DeprecationWarning: 'language_model_type' is deprecated and will be removed in v2.0.0. Use model, config, or model_id parameters instead.
  result =  lx.extract(


[14:25:00] INFO     Starting document annotation.                                                 ]8;id=64984;file:///Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/.venv/lib/python3.10/site-packages/langextract/annotation.py\annotation.py]8;;\:]8;id=552274;file:///Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/.venv/lib/python3.10/site-packages/langextract/annotation.py#261\261]8;;\

           INFO     Processing batch 0 with length 10                                             ]8;id=219591;file:///Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/.venv/lib/python3.10/site-packages/langextract/annotation.py\annotation.py]8;;\:]8;id=685958;file:///Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/.venv/lib/python3.10/site-packages/langextract/annotation.py#282\282]8;;\

[14:25:04] INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK" ]8;id=766610;file:///Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/.venv/lib/python3.10/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=689339;file:///Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/.venv/lib/python3.10/site-packages/httpx/_client.py#1025\1025]8;;\

           INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK" ]8;id=532306;file:///Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/.venv/lib/python3.10/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=183217;file:///Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/.venv/lib/python3.10/site-packages/httpx/_client.py#1025\1025]8;;\

...

           INFO     Retrying request to /chat/completions in 4.676000 seconds                  ]8;id=835797;file:///Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/.venv/lib/python3.10/site-packages/openai/_base_client.py\_base_client.py]8;;\:]8;id=452220;file:///Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/.venv/lib/python3.10/site-packages/openai/_base_client.py#1071\1071]8;;\

[14:25:21] INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too ]8;id=416439;file:///Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/.venv/lib/python3.10/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=563918;file:///Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/.venv/lib/python3.10/site-packages/httpx/_client.py#1025\1025]8;;\
                    Many Requests"                                                                                 

[14:25:22] INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too ]8;id=82945;file:///Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/.venv/lib/python3.10/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=647634;file:///Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/.venv/lib/python3.10/site-packages/httpx/_client.py#1025\1025]8;;\
                    Many Requests"                                                                                 

           INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too ]8;id=535528;file:///Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/.venv/lib/python3.10/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=525137;file:///Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/.venv/lib/python3.10/site-packages/httpx/_client.py#1025\1025]8;;\
                    Many Requests"                                                                                 

InferenceRuntimeError: Parallel inference error: OpenAI API error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-hVUeXdZ0q4N17L4ItlDla2ep on tokens per min (TPM): Limit 30000, Used 30000, Requested 2121. Please try again in 4.242s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}

In [31]:
joined_texts.keys()

dict_keys(['document-06.docx', 'documento-entrerios-03.docx', 'documento-entrerios-02.docx', 'document-07.docx', 'document-02.docx', 'document-03.docx', 'document-04.docx', 'document-08.docx'])

In [34]:
import pickle
with open("results_openai-entrerios0203.pkl", "wb") as f:
    pickle.dump(results, f)

In [35]:
import pickle
with open("predictions_openai-entrerios0203.pkl", "wb") as f:
    pickle.dump(predictions, f)

#### Multiple docs

In [ ]:
import time

start = timeit.timeit()
predictions = {}
results = {}

for name_doc, text in joined_texts.items():
    print(name_doc)
    results[name_doc], predictions[name_doc] = langextract_prediction(text, PROMPT, examples, openai_api_key)
    time.sleep(25)  # Sleep for 2 seconds to avoid rate limit

end = timeit.timeit()
print('\n Total time: ', end-start)


In [41]:
joined_texts['documento-entrerios-02.docx'][555:577]


'Ana Carolina Rodríguez'

In [37]:
predictions['documento-entrerios-02.docx']

[{'label': 'LOC',
  'text': 'San Lorenzo',
  'start_char': 41,
  'end_char': 52,
  'attrs': {},
  'alignment_status': <AlignmentStatus.MATCH_EXACT: 'match_exact'>},
 {'label': 'LOC',
  'text': 'Provincia de Santa Fe',
  'start_char': 201,
  'end_char': 222,
  'attrs': {},
  'alignment_status': <AlignmentStatus.MATCH_EXACT: 'match_exact'>},
 {'label': 'FECHA',
  'text': '22 días del mes de noviembre de 2023',
  'start_char': 230,
  'end_char': 266,
  'attrs': {},
  'alignment_status': <AlignmentStatus.MATCH_EXACT: 'match_exact'>},
 {'label': 'PER',
  'text': 'Verónica Salvatierra',
  'start_char': 322,
  'end_char': 342,
  'attrs': {},
  'alignment_status': <AlignmentStatus.MATCH_EXACT: 'match_exact'>},
 {'label': 'PER',
  'text': 'Ana Carolina Rodríguez',
  'start_char': 555,
  'end_char': 577,
  'attrs': {},
  'alignment_status': <AlignmentStatus.MATCH_FUZZY: 'match_fuzzy'>},
 {'label': 'PER',
  'text': 'Diego Esteban Fernández',
  'start_char': 690,
  'end_char': 713,
  'attrs': {},


In [ ]:
import pickle
with open("predictions_openai-entrerios0203.pkl", "wb") as f:
    pickle.dump(predictions, f)

### Calculate metrics

In [ ]:
import pickle
with open("predictions_openai-alldocs.pkl", "wb") as f:
    pickle.dump(predictions, f)

In [ ]:
dfs = []
for doc, start_end_chars in doc_start_end_chars.items():
    df_chars = pd.DataFrame(start_end_chars)
    df_chars['doc'] = doc
    dfs.append(df_chars)

all_df_chars = pd.concat(dfs, ignore_index=True)

main_df = df.merge(
    all_df_chars,
    left_on=['paragraph_id', 'name'],
    right_on=['paragraph_id', 'doc'],
    how='inner'
)
main_df.to_csv('documents-02-08-sin05-conOpenAI.csv', index=False)

In [25]:
results.keys()

dict_keys(['documento-entrerios-02.docx', 'documento-entrerios-03.docx'])

Interactive view of predictions:

In [22]:
result = results['documento-entrerios-03.docx']
extractions_name = 'extractions-entrerios-03.json'
extractions_html = 'extractions-entrerios-03.html'

print(f"Extracted {len(result.extractions)} entities from {len(result.text):,} characters")

# Save and visualize the results
lx.io.save_annotated_documents([result], output_name=extractions_name, output_dir=".")

# Generate the interactive visualization
html_content = lx.visualize(extractions_name)

with open(extractions_html, "w") as f:
    if hasattr(html_content, 'data'):
        f.write(html_content.data)  # For Jupyter/Colab
    else:
        f.write(html_content)

print("Interactive visualization saved to test_extractions.html")

Extracted 26 entities from 3,818 characters


LangExtract: Saving to extractions-entrerios-03.json: 1 docs [00:00, 419.30 docs/s]

✓ Saved 1 documents to extractions-entrerios-03.json



LangExtract: Loading extractions-entrerios-03.json: 100%|██████████| 10.2k/10.2k [00:00<00:00, 6.65MB/s]

✓ Loaded 1 documents from extractions-entrerios-03.json
Interactive visualization saved to test_extractions.html


In [ ]:
main_df.head(4)

# Assuming ordered paragraphs

In [ ]:
for i in range(len(result.extractions)):
    print(f"{i+1}: \n CLASS: {result.extractions[i].extraction_class} \n TEXT: {result.extractions[i].extraction_text} \n from_chars: {text[result.extractions[i].char_interval.start_pos:result.extractions[i].char_interval.end_pos]}")

### LLama3.2

In [ ]:
'''
result = lx.extract(
    text_or_documents=text,
    prompt_description=PROMPT,
    examples=examples,
    language_model_type=lx.inference.OllamaLanguageModel,
    model_id="llama3.2:3b",
    model_url="http://host.docker.internal:11434",
    max_char_buffer=1000,
    extraction_passes=1,
    max_workers=6,
    fence_output=False,
    use_schema_constraints=False,
    language_model_params={
        "temperature": 0.1,          # menos creatividad - > checkear que con t=0 sea determinista
        "top_p": 0.9,
        "top_k": 40,
        "max_output_tokens": 400,
        "timeout": 600,
        "keep_alive": "10m",
        "num_ctx": 4096,             # cuántos tokens de contexto
    },
)
'''

# Old prompt

In [ ]:
# 1. Define the prompt and extraction rules
prompt = """
    Sos un asistente especializado en el análisis de documentos judiciales.
    Tu tarea es identificar y extraer menciones de información sensible para su posterior anonimización.
    Debés detectar fragmentos textuales que correspondan a cualquiera de las siguientes entidades:

    - "BANCO": Nombre de una entidad bancaria, pública o privada.
    - "CBU": Código Bancario Uniforme (22 dígitos) de una cuenta.
    - "CORREO_ELECTRONICO": Dirección de correo electrónico.
    - "CUIJ": Código Único de Identificación Jurídica de causas judiciales.
    - "CUIT_CUIL": Número de CUIT o CUIL de una persona física o jurídica.
    - "DIRECCION": Dirección postal específica (calle, número, etc.).
    - "DNI": Número de Documento Nacional de Identidad u otro documento identificatorio.
    - "EDAD": Edad explícita de una persona.
    - "ESTUDIOS": Nivel o institución educativa que permita identificar a la persona (ej. "primario incompleto", "secundario completo", "Licenciado en…").
    - "FECHA": Fecha completa o parcial (día, mes y/o año).
    - "LINK": Enlace o URL a una página web.
    - "LOC": Localización geográfica específica (ciudad, barrio, comisaría, etc.).
    - "MARCA_AUTOMOVIL": Marca de un vehículo (ej. Toyota, Ford).
    - "NACIONALIDAD": Nacionalidad de una persona (ej. "argentino", "brasileña").
    - "NUM_ACTUACION": Número identificatorio de una actuación administrativa o contravencional.
    - "NUM_CAJA_AHORRO": Número completo de una caja de ahorro o cuenta bancaria.
    - "NUM_EXPEDIENTE": Número de expediente judicial o administrativo.
    - "NUM_MATRICULA": Número de matrícula profesional o académica.
    - "PATENTE_DOMINIO": Patente o dominio de un vehículo.
    - "PER": Nombre y apellido(s) de una persona física. Los nombres inicializados y los apodos también cuentan como información sensible a anonimizar.
    - "TELEFONO": Número telefónico (fijo o celular).
"""

In [ ]:
prompt = textwrap.dedent("""
Eres un asistente que extrae ENTIDADES sensibles para anonimización en documentos judiciales en español.
Reglas:
- Usa TEXTO EXACTO del documento (no parafrasees).
- No superpongas entidades; una mención = una extracción.
- Si tenés dudas, no inventes.
Clases permitidas y guía breve:
- BANCO: nombre de entidad bancaria.
- CBU: 22 dígitos continuos.
- CORREO_ELECTRONICO: formato correo válido.
- CUIJ: código causa judicial (ej. 12-34567890-1).
- CUIT_CUIL: CUIT/CUIL (##-########-#).
- DIRECCION: calle y número (opcionalmente ciudad/barrio).
- DNI: número de documento (solo dígitos).
- EDAD: número de edad explícito (en años).
- ESTUDIOS: nivel/institución educativa que identifique a la persona.
- FECHA: dd/mm/aaaa, dd-mm-aaaa o “5 de mayo de 2023”.
- LINK: URL http/https.
- LOC: localidad/barrio/comisaría/ciudad.
- MARCA_AUTOMOVIL: marca (Toyota, Ford, Volkswagen, etc.).
- NACIONALIDAD: gentilicio (argentino, paraguaya, ...).
- NUM_ACTUACION: nro. de actuación administrativa/contravencional.
- NUM_CAJA_AHORRO: número de caja de ahorro/cuenta.
- NUM_EXPEDIENTE: nro. de expediente.
- NUM_MATRICULA: matrícula profesional o académica.
- PATENTE_DOMINIO: dominio vehicular (p. ej., AB123CD).
- PER: nombre(s) y apellido(s) de persona física. También apodos/iniciales.

Salida: solo las entidades que estén explícitas en el texto.
""").strip()

In [ ]:
# 2. Provide a high-quality example to guide the model
examples = [
    lx.data.ExampleData(
        text="El 5 de mayo de 2023 el señor Fiscal indicó que realizó distintas medidas de prueba y que del resultado surge que tanto la investigada como el menor Juan Pérez se domicilian en la calle Sarmiento 1234, de la localidad de Moreno, por lo que solicitó que se declare la incompetencia en razón del territorio y se envíe el caso al Juzgado de Garantías que corresponda del Departamento Judicial de Moreno, con jurisdicción en el partido de Moreno.",
        extractions=[
            lx.data.Extraction(
                extraction_class="FECHA", extraction_text="5 de mayo de 2023"
            ),
            lx.data.Extraction(
                extraction_class="PER", extraction_text="Juan Pérez"
            ),
            lx.data.Extraction(
                extraction_class="DIRECCION", extraction_text="Sarmiento 1234"
            ),
            lx.data.Extraction(
                extraction_class="LOC", extraction_text="Moreno"
            ),
        ],
    ),
    lx.data.ExampleData(
        text="JUZGADO DE 1RA INSTANCIA EN LO PENAL CONTRAVENCIONAL Y DE FALTAS N°10 SECRETARIA N°19\nCarlos Gómez sobre 84 - HOMICIDIO CULPOSO Y OTROS\nNúmero: 52345/2022\nCUIJ: 12-34567890-1\nActuación Nro: 2022-009876",
        extractions=[
            lx.data.Extraction(
                extraction_class="PER", extraction_text="Carlos Gómez"
            ),
            lx.data.Extraction(
                extraction_class="NUM_EXPEDIENTE", extraction_text="52345/2022"
            ),
            lx.data.Extraction(
                extraction_class="CUIJ", extraction_text="12-34567890-1"
            ),
            lx.data.Extraction(
                extraction_class="NUM_ACTUACION", extraction_text="2022-009876"
            ),
        ],
    ),
    lx.data.ExampleData(
        text="Acusado: Miguel Torres, DNI 30123456, nacido el 14/02/1990, de 34 años de edad, de nacionalidad paraguaya, género varón cis, con estudios secundarios completos, hizo hasta 3er año porque fue padre joven, con último domicilio en Av. Corrientes 3456, de esta ciudad, donde vive con su hermana y su cuñado Jorge Pérez. Tiene dos hijos a su cargo, de 5 y 8 años. Su hijo de 8 vive con él, su hija de 5 vive con su madre, Laura Fernández.",
        extractions=[
            lx.data.Extraction(
                extraction_class="PER", extraction_text="Miguel Torres"
            ),
            lx.data.Extraction(
                extraction_class="DNI", extraction_text="30123456"
            ),
            lx.data.Extraction(
                extraction_class="FECHA", extraction_text="14/02/1990"
            ),
            lx.data.Extraction(extraction_class="EDAD", extraction_text="34"),
            lx.data.Extraction(
                extraction_class="NACIONALIDAD", extraction_text="paraguaya"
            ),
            lx.data.Extraction(
                extraction_class="ESTUDIOS",
                extraction_text="estudios secundarios completos",
            ),
            lx.data.Extraction(
                extraction_class="DIRECCION",
                extraction_text="Av. Corrientes 3456",
            ),
            lx.data.Extraction(
                extraction_class="PER", extraction_text="Jorge Pérez"
            ),
            lx.data.Extraction(extraction_class="EDAD", extraction_text="5"),
            lx.data.Extraction(extraction_class="EDAD", extraction_text="8"),
            lx.data.Extraction(
                extraction_class="PER", extraction_text="Laura Fernández"
            ),
        ],
    ),
    lx.data.ExampleData(
        text="El testigo Juan López dejó asentado su número de contacto: 11-2345-6789. Indicó que la médica Dra. Ana García, MN 12345, asistió al lugar donde se hallaba un vehículo Volkswagen, patente AB123CD.",
        extractions=[
            lx.data.Extraction(
                extraction_class="PER", extraction_text="Juan López"
            ),
            lx.data.Extraction(
                extraction_class="TELEFONO", extraction_text="11-2345-6789"
            ),
            lx.data.Extraction(
                extraction_class="PER", extraction_text="Ana García"
            ),
            lx.data.Extraction(
                extraction_class="NUM_MATRICULA", extraction_text="12345"
            ),
            lx.data.Extraction(
                extraction_class="MARCA_AUTOMOVIL",
                extraction_text="Volkswagen",
            ),
            lx.data.Extraction(
                extraction_class="PATENTE_DOMINIO", extraction_text="AB123CD"
            ),
        ],
    ),
    lx.data.ExampleData(
        text="Se identificó una transferencia bancaria con los siguientes datos: CUIT 20-12345678-3, CBU 2850590940090412345671, Caja de Ahorro N° 12345678, Banco Nación.",
        extractions=[
            lx.data.Extraction(
                extraction_class="CUIT_CUIL", extraction_text="20-12345678-3"
            ),
            lx.data.Extraction(
                extraction_class="CBU",
                extraction_text="2850590940090412345671",
            ),
            lx.data.Extraction(
                extraction_class="NUM_CAJA_AHORRO", extraction_text="12345678"
            ),
            lx.data.Extraction(
                extraction_class="BANCO", extraction_text="Banco Nación"
            ),
        ],
    ),
    lx.data.ExampleData(
        text="Para mayor información, comunicarse a fiscalia.central@justicia.gob.ar o visitar el sitio https://justicia.gob.ar/actuaciones.",
        extractions=[
            lx.data.Extraction(
                extraction_class="CORREO_ELECTRONICO",
                extraction_text="fiscalia.central@justicia.gob.ar",
            ),
            lx.data.Extraction(
                extraction_class="LINK",
                extraction_text="https://justicia.gob.ar/actuaciones",
            ),
        ],
    ),
]

In [ ]:
text = "La Fiscalía determinó que el objeto de este caso es investigar el hecho que tuvo lugar el día 12 de marzo de 2023 a las 8:50 horas aproximadamente, ocasión en que Carlos Gómez y María Rodriguez estafaron a Juan Pérez por un monto total de pesos treinta y tres mil novecientos sesenta ($33.960)."

# Run the extraction
result = lx.extract(
    text_or_documents=text,
    prompt_description=prompt,
    examples=examples,
    model_id="llama3.2:3b",
    model_url="http://host.docker.internal:11434",
    max_workers=20,
    fence_output=False,
    language_model_params={
            "timeout": 600,       # aumentar el timeout a 10 minutos
            "keep_alive": "5m"    # mantener modelo cargado 5 minutos
        },
    use_schema_constraints=False,
)

In [ ]:
print(text)

In [ ]:
pprint(result)